In [ ]:
import numpy as np
from scipy.constants import N_A
import pandas as pd
import os
from numpy.linalg import norm

# Constants
PI = np.pi

# Define a mapping between atomic symbols and van der Waals radii.
ATOMIC_SYMBOLS = [
    "H", "He", "Li", "Be", "B", "C", "N", "O", "F", "Ne", "Na", "Mg", "Al", "Si", "P", "S",
    "Cl", "Ar", "K", "Ca", "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Ga", "Ge",
    "As", "Se", "Br", "Kr", "Rb", "Sr", "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",
    "In", "Sn", "Sb", "Te", "I", "Xe", "Cs", "Ba", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd",
    "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg",
    "Tl", "Pb", "Bi", "Po", "At", "Rn", "Fr", "Ra", "Ac", "Th", "Pa", "U", "Np", "Pu", "Am", "Cm",
    "Bk", "Cf", "Es", "Fm", "Md", "No", "Lr", "Rf", "Db", "Sg", "Bh", "Hs", "Mt", "Ds", "Rg", "Cn",
    "Nh", "Fl", "Mc", "Lv", "Ts", "Og"
]

BONDI_VDW_RADIUS = [
    1.10, 1.40, 1.81, 1.53, 1.92, 1.70, 1.55, 1.52, 1.47, 1.54, 2.27, 1.73, 1.84, 2.10, 1.80, 1.80,
    1.75, 1.88, 2.75, 2.31, 2.30, 2.15, 2.05, 2.05, 2.05, 2.05, 2.00, 2.00, 2.00, 2.10, 1.87, 2.11,
    1.85, 1.90, 1.83, 2.02, 3.03, 2.49, 2.40, 2.30, 2.15, 2.10, 2.05, 2.05, 2.00, 2.05, 2.10, 2.20,
    1.93, 2.17, 2.06, 2.06, 1.98, 2.16, 3.43, 2.68, 2.50, 2.48, 2.47, 2.45, 2.43, 2.42, 2.40, 2.38,
    2.37, 2.35, 2.33, 2.32, 2.30, 2.28, 2.27, 2.25, 2.20, 2.10, 2.05, 2.00, 2.00, 2.05, 2.10, 2.05,
    1.96, 2.02, 2.07, 1.97, 2.02, 2.20, 3.48, 2.83, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40,
    2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40,
    2.40, 2.40, 2.40, 2.40, 2.40, 2.40
]

ELEMENT_TO_RADIUS = {symbol: radius for symbol, radius in zip(ATOMIC_SYMBOLS, BONDI_VDW_RADIUS)}

def bondi_vdW(element):
    """Get the van der Waals radii of atoms"""
    try:
        return ELEMENT_TO_RADIUS[element]
    except KeyError:
        raise ValueError(f"Error: van der Waals radius for element {element} not found")

def sphere_volume(r):
    """Calculate the volume of sphere"""
    return (4 / 3) * PI * r**3

def intersection_volume(r1, r2, d):
    """Calculate the intersection volume between two spheres"""
    if d >= r1 + r2:
        return 0.0
    else:
        return (PI * (r1 + r2 - d)**2 * (d**2 + 2 * d * (r1 + r2) - 3 * (r1 - r2)**2)) / (12 * d)

def calculate_vdw_volume(xyz_file_path):
    """Calculate the van der Waals volume of molecules"""
    # Read XYZ file
    with open(xyz_file_path, 'r') as f:
        try:
            n_atom = int(f.readline().strip())  # Read the number of atoms
        except ValueError:
            raise ValueError("Invalid XYZ file: The first line must be the number of atoms.")

        f.readline()  # Skip the comment line

        elements = []
        xyz = []
        for line_num, line in enumerate(f, start=3):  # Start counting from line 3
            parts = line.split()
            if len(parts) != 4:
                print(f"Warning: Line {line_num} is invalid (expected 4 fields, got {len(parts)}). Skipping this line.")
                continue  # Skip invalid lines
            try:
                elements.append(parts[0])
                xyz.append([float(parts[1]), float(parts[2]), float(parts[3])])
            except ValueError as e:
                print(f"Warning: Line {line_num} contains invalid data. Skipping this line. Error: {e}")
                continue  # Skip lines with invalid data

    if len(elements) != n_atom:
        raise ValueError(f"Invalid XYZ file: Expected {n_atom} atoms, but found {len(elements)} atoms.")

    elements = np.array(elements)
    xyz = np.array(xyz)
    n_atom = len(elements)
    # Get the van der Waals radii
    radii = np.array([bondi_vdW(element) for element in elements])

    volume = 0.0
    for i in range(n_atom):
        volume += sphere_volume(radii[i])
        for j in range(i):
            d = norm(xyz[i] - xyz[j])  # Distance between atoms i and j
            volume -= intersection_volume(radii[i], radii[j], d)
    return volume

# Calculate the molecular mass
def calculate_molecular_mass(atoms):
    atomic_masses = {'H': 1.008, 'O': 15.999, 'C': 12.011, 'N': 14.007, 'Hf': 178.49}   
    total_mass = sum(atomic_masses.get(atom, 0) for atom in atoms)  
    return total_mass

# Calculate the center of mass
def calculate_com(atoms, coords):
    atomic_masses = {'H': 1.008, 'O': 15.999, 'C': 12.011, 'N': 14.007, 'Hf': 178.49}
    masses = np.array([atomic_masses[atom] for atom in atoms])
    com = np.sum(masses[:, None] * coords, axis=0) / np.sum(masses)
    return com

# Calculate the inertia tensor
def calculate_inertia_tensor(atoms, coords, com):
    atomic_masses = {'H': 1.008, 'O': 15.999, 'C': 12.011, 'N': 14.007, 'Hf': 178.49}
    masses = np.array([atomic_masses[atom] for atom in atoms])
    coords_centered = coords - com
    I = np.zeros((3, 3))
    for i in range(len(atoms)):
        x, y, z = coords_centered[i]
        m = masses[i]
        I[0, 0] += m * (y**2 + z**2)
        I[1, 1] += m * (x**2 + z**2)
        I[2, 2] += m * (x**2 + y**2)
        I[0, 1] -= m * x * y
        I[1, 0] -= m * x * y
        I[0, 2] -= m * x * z
        I[2, 0] -= m * x * z
        I[1, 2] -= m * y * z
        I[2, 1] -= m * y * z
    return I

# Calculate the square of the radius of gyration (r_g^2)
def calculate_rg_squared(atoms, coords):
    num_atoms = len(atoms)
    # Calculate the center of mass
    com = calculate_com(atoms, coords)
    # Calculate r_g^2
    rg_squared = 0.0
    for i in range(num_atoms):
        r_k = coords[i]
        rg_squared += np.linalg.norm(r_k - com) ** 2
    rg_squared /= num_atoms
    return rg_squared

def main():
    # The specified folder path
    folder_path = r'mol_xyz_files'
    
    # Get all xyz files
    xyz_files = [f for f in os.listdir(folder_path) if f.endswith('.xyz')]

    # Initialize a dictionary to store results
    results = {}
    
    for xyz_file in xyz_files:
        file_path = os.path.join(folder_path, xyz_file)
        print(f"Processing file: {xyz_file}")
        
        with open(file_path, 'r') as f:
            lines = f.readlines()
        num_atoms = int(lines[0].strip())
        atoms = []
        coords = []
        for line in lines[2:2+num_atoms]:
            parts = line.strip().split()
            atoms.append(parts[0])
            coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
        coords = np.array(coords)
        
        # Calculate the molecular mass
        molecular_mass = calculate_molecular_mass(atoms)
        print(f"Molecular Mass: {molecular_mass} g/mol")
        
        # Calculate the inertia tensor
        com = calculate_com(atoms, coords)
        inertia_tensor = calculate_inertia_tensor(atoms, coords, com)
        eigenvalues = np.linalg.eigvals(inertia_tensor)
        I = np.prod(eigenvalues)**(1/3)
        I = I * 1e-3 / N_A * 1e-20
        print(f"Average Moment of Inertia I = {I}")
        
        # Calculate the radius of gyration
        rg_squared = calculate_rg_squared(atoms, coords)
        r_g = np.sqrt(rg_squared)
        print(f"Radius of Gyration r_g: {r_g}")
        
        # Calculate the van der Waals volume
        volume = calculate_vdw_volume(file_path)
        print(f"Van der Waals Volume (Å^3): {volume:.8f}")
        print("-" * 40)

        # Store results
        results[xyz_file] = {
            'Molecular Mass (g/mol)': molecular_mass,
            'Average Moment of Inertia (I)': I,
            'Radius of Gyration (r_g)': r_g,
            'Van der Waals Volume (Å^3)': volume
        }
    
    # Convert results to a DataFrame
    df = pd.DataFrame.from_dict(results, orient='index')
    df.index.name = 'File Name'
    
    # Save to CSV
    output_csv = 'molecular_properties.csv'
    df.to_csv(output_csv)
    print(f"Results saved to {output_csv}")

if __name__ == "__main__":
    main()